# Eval Trained Adapter v12 — Diagnostic on Stratified Holdout (Kh0a-recipe Calibration LB=0.72)

Loads the LoRA adapter trained in `gideonnazarian/v12-kh0a-baseline-calibration` (output: `nemotron-lora-adapter/`) onto Nemotron 3 Nano and runs inference on a stratified per-category holdout to diagnose where the v12 model fails.

**Why this exists:** v12 = fork of Kh0a's public 0.87-cluster recipe with our v3 bit_manipulation solver patched in (Kh0a's public notebook had a copy-paste bug where `generate_bit_manipulation_trace` was missing — their private cached CSV hid the bug). v12 scored **LB 0.72** on submission. That's **+0.10 vs our v9 HWM (0.62)** but **-0.13 vs Kh0a's claimed 0.87**. The gap is confounded between (a) recipe-reproducibility, (b) our bit_manip substitute quality, (c) missing private val split, (d) test set drift.

**This eval is Experiment A from Part G of the intel doc** — diagnostic eval to see per-category accuracy. If bit_manipulation is dramatically worse than other categories → our substitute is the dominant problem. If all categories are similarly impacted → recipe-reproducibility issue (not our patch).

**Holdout:** stratified 5 problems per category × 6 categories = 30 problems, fixed seed=42. Avoids the last-5-by-id pattern used for v6-v9 (we want category breakdown, not direct comparability to v9). The 5-problem comparison was useful when iterating from a 0.5x baseline; now we need real per-category signal.

**What to look for:**
- Per-category strict accuracy. If `bit_manipulation` ≪ others → our substitute is hurting; v13 should improve bit_manip data quality (distilled traces from a stronger model).
- If `cipher` is bad → Kh0a's cipher trace generator is weaker than ours. Swap in our cipher v2 (wordlist fallback) for v13.
- If `numeric_equation`/`symbol_transform` is bad → Kh0a's category-rebalancing didn't transfer; v13 might need to drop these or replace with our transformation_rules solver.
- If everything is uniformly mediocre → recipe-reproducibility issue (timing, val split, seed). v13 unlikely to help much without addressing root cause.

**Reference:**
- v9 eval (v3 adapter): 2/5 strict, 3/5 substring (LB 0.58)
- v9 eval (v9 adapter): 3/5 strict, 4/5 substring (LB 0.62)
- v12 calibration: **LB 0.72** (THIS adapter)

In [ ]:
# Cell 1: Setup (sitedir + ptxas) — same boilerplate as other kernels
import os, random, shutil, site, stat, sys, time, glob
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Load base model + tokenizer + ATTACH v12 trained adapter via PeftModel
import kagglehub
import mamba_ssm  # noqa: F401
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"Tokenizer loaded; IM_END_ID={IM_END_ID}")

# v12 adapter lives in the v12 training kernel's output directory
# (Kh0a's notebook saves to ./nemotron-lora-adapter/ which becomes kernel-output path)
ADAPTER_CANDIDATES = [
    "/kaggle/input/notebooks/gideonnazarian/v12-kh0a-baseline-calibration/nemotron-lora-adapter",
    "/kaggle/input/v12-kh0a-baseline-calibration/nemotron-lora-adapter",
]
ADAPTER_PATH = None
for p in ADAPTER_CANDIDATES:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "adapter_config.json")):
        ADAPTER_PATH = p
        break
if ADAPTER_PATH is None:
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        ADAPTER_PATH = os.path.dirname(hits[0])
if ADAPTER_PATH is None:
    raise FileNotFoundError(f"adapter_config.json not found. /kaggle/input/ contains: {os.listdir('/kaggle/input')}")
step(f"ADAPTER_PATH={ADAPTER_PATH}")

t = time.perf_counter()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Base model loaded in {time.perf_counter() - t:.0f}s")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
step(f"v12 adapter attached via PeftModel.from_pretrained")
step(f"VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 3: Stratified per-category eval (5 per category × 6 categories = 30 problems)
import re
import json
import pandas as pd
import random

random.seed(42)

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)
SAMPLES_PER_CATEGORY = 5
MAX_NEW_TOKENS = 4096   # Kh0a's training used MAX_SEQ_LEN=3500, so generation up to ~4k is safe

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV)

# Classify each problem by its prompt (matches Kh0a's classify_prompt logic)
def classify(prompt: str) -> str:
    p = prompt.lower()
    if "bit manipulation" in p: return "bit_manipulation"
    if "secret encryption rules" in p: return "cipher"
    if "unit conversion" in p: return "unit_conversion"
    if "numeral system" in p: return "numeral_system"
    if "gravitational constant" in p: return "gravity"
    if "transformation rules" in p: return "transformation_rules"
    return "unknown"

df["category"] = df["prompt"].apply(classify)
print("Category counts:", df["category"].value_counts().to_dict())

# Stratified sample
holdout_ids = []
for cat in sorted(df["category"].unique()):
    if cat == "unknown": continue
    cat_df = df[df["category"] == cat]
    n = min(SAMPLES_PER_CATEGORY, len(cat_df))
    sampled = cat_df.sample(n=n, random_state=42)
    holdout_ids.extend(sampled["id"].tolist())
holdout = df[df["id"].isin(holdout_ids)].copy()
step(f"Stratified holdout: {len(holdout)} problems across {holdout['category'].nunique()} categories")
for cat in sorted(holdout["category"].unique()):
    print(f"  {cat}: {len(holdout[holdout['category']==cat])}")

BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
THINK_CLOSE = "</think>"
IM_END = "<|im_end|>"

def extract_after_think(c):
    if THINK_CLOSE not in c: return ""
    post = c.split(THINK_CLOSE, 1)[1]
    if IM_END in post: post = post.split(IM_END, 1)[0]
    return post.strip()

def extract_boxed_anywhere(c):
    matches = list(BOXED_RE.finditer(c))
    return matches[-1].group(1).strip() if matches else ""

def gold_substring(c, g):
    return g if (g and g in c) else ""

results = []
for i, row in holdout.iterrows():
    pid = row["id"]; gold = str(row["answer"]); cat = row["category"]
    prompt_text = row["prompt"] + PROMPT_SUFFIX
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        return_tensors="pt", add_generation_prompt=True, return_dict=True,
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    t_p = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            eos_token_id=[IM_END_ID, tokenizer.eos_token_id],
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t_p
    n_new = out.shape[1] - prompt_len
    completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=False)
    after = extract_after_think(completion)
    boxed = extract_boxed_anywhere(completion)
    substr = gold_substring(completion, gold)
    results.append({
        "id": pid, "category": cat, "completion_tokens": n_new, "seconds": elapsed,
        "after_think": after, "boxed_anywhere": boxed, "gold": gold,
        "completion": completion,
        "correct_after_think": after.strip() == gold.strip(),
        "correct_boxed": boxed.strip() == gold.strip(),
        "correct_substring": bool(substr),
        "has_think_close": THINK_CLOSE in completion,
    })
    mark = "✓" if boxed.strip()==gold.strip() else ("~" if substr else "✗")
    print(
        f"  [{len(results):2d}/{len(holdout)}] {cat:22s} {pid} | {n_new:4d} tok in {elapsed:5.1f}s | "
        f"boxed={boxed!r:25s} gold={gold!r:15s} {mark}",
        flush=True,
    )

In [ ]:
# Cell 4: Per-category summary + save for offline analysis
import json
import pandas as pd
from collections import defaultdict

n = len(results)
print("\n=== v12 Adapter (Kh0a recipe + our bit_manip patch, LB=0.72) — Stratified Eval ===")
print(f"Total: {n} problems")

# Aggregate per category
per_cat = defaultdict(lambda: {"n": 0, "strict": 0, "substring": 0, "after_think": 0, "close": 0})
for r in results:
    c = r["category"]
    per_cat[c]["n"] += 1
    per_cat[c]["strict"] += int(r["correct_boxed"])
    per_cat[c]["substring"] += int(r["correct_substring"])
    per_cat[c]["after_think"] += int(r["correct_after_think"])
    per_cat[c]["close"] += int(r["has_think_close"])

print(f"\n{'category':22s} | strict  | substring | after_think | think_close")
print("-" * 80)
for cat in sorted(per_cat):
    s = per_cat[cat]
    print(
        f"{cat:22s} | {s['strict']}/{s['n']}    | "
        f"{s['substring']}/{s['n']}      | "
        f"{s['after_think']}/{s['n']}        | "
        f"{s['close']}/{s['n']}"
    )

agg_strict = sum(s["strict"] for s in per_cat.values()) / n
agg_substr = sum(s["substring"] for s in per_cat.values()) / n
print(f"\nAggregate: strict={agg_strict:.2%}, substring={agg_substr:.2%}")

print("\n=== Reference ===")
print("v9 eval (v3 adapter, LB 0.58): strict 2/5=40%, substring 3/5=60% (5-problem holdout)")
print("v9 eval (v9 adapter, LB 0.62): strict 3/5=60%, substring 4/5=80% (5-problem holdout)")
print(f"v12 (this adapter, LB 0.72):  strict {agg_strict:.0%}, substring {agg_substr:.0%} (30-problem stratified)")

# Save
pd.DataFrame(results).to_csv("/kaggle/working/v12_eval_results.csv", index=False)
with open("/kaggle/working/v12_eval_summary.json", "w") as f:
    json.dump({
        "n_total": n,
        "agg_strict": agg_strict,
        "agg_substring": agg_substr,
        "per_category": dict(per_cat),
    }, f, indent=2)
step(f"Saved v12_eval_results.csv + v12_eval_summary.json")

In [ ]:
# Cell 5: Last 800 chars per problem for offline failure-mode analysis
for i, r in enumerate(results, 1):
    print(f"=== [{i:2d}/{len(results)}] {r['category']:22s} {r['id']} gold={r['gold']!r} ===")
    print(f"close={r['has_think_close']} | boxed={r['boxed_anywhere']!r} | substr={r['correct_substring']} | strict={r['correct_boxed']}")
    print("--- LAST 800 CHARS ---")
    print(r["completion"][-800:])
    print()
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")